In [1]:
!pip install corus
!python3 -m spacy download ru_core_news_sm
!pip install gensim
!pip install razdel
!pip install navec
!pip install natasha
!pip install datasets evaluate seqeval transformers accelerate sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 41.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 7.1 MB/s eta 0:

In [2]:
import os, re, math, random, gc, json

import numpy as np
import pandas as pd

import torch
from torch import nn
import torch.nn.functional as F

from datasets import load_dataset, DatasetDict, Dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    pipeline
)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

from natasha import Doc, Segmenter, NewsEmbedding, NewsNERTagger
from razdel import sentenize

from huggingface_hub import login
from google.colab import userdata

In [3]:
data = load_dataset("gusevski/factrueval2016")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train_data.json:   0%|          | 0.00/7.62M [00:00<?, ?B/s]

dev_data.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

test_data.json:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

Сразу сохраню в виде json все токенизированные тексты из трейн сплита, чтобы на их основе добавить в модель лексики с помощью другой LLM для дальнейшего повышения качества на в NER

In [ ]:
full_text = []

for i in range(len(data['data'])):
  sup = data['data'][i]['tokens']
  full_text.append(' '.join(sup).strip())

In [ ]:
with open('tokens.json', 'w', encoding='utf-8') as f:
    json.dump(full_text, f, ensure_ascii=False, indent=4)

In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

Замечу, что HF датасет очень странно завернут и для доступа к данным(токенам и лейблам) нужно войти еще на несколько уровней вглубь. В связи с этим выполним переупаковку датасета, чтобы все сэмплы лежали на верхнем уровне и доступ был к ним в одно действие

In [ ]:
data["train"].features

{'data': List({'id': Value('int64'), 'tokens': List(Value('string')), 'length': Value('int64'), 'ner_tags_str': List(Value('string')), 'ner_tags': List(Value('int64'))})}

In [4]:
def unpack_dataset(dataset_dict):
    new_dataset_dict = {}

    for split_name, split_dataset in dataset_dict.items():
        unpacked_data = []

        for item_list in split_dataset['data']:
            for item in item_list:
                unpacked_item = {
                    'tokens': item['tokens'],
                    'ner_tags': item['ner_tags'],
                    'ner_tags_str': item.get('ner_tags_str', []),
                    'id': item.get('id', -1),
                    'length': item.get('length', 0)
                }
                unpacked_data.append(unpacked_item)

        new_dataset_dict[split_name] = Dataset.from_list(unpacked_data)

    return DatasetDict(new_dataset_dict)

unpacked_dataset = unpack_dataset(data)

print(unpacked_dataset)
print(unpacked_dataset['train'].features)
print(f"Количество строк в train: {len(unpacked_dataset['train'])}")
print(unpacked_dataset['train'][0].keys())

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'ner_tags_str', 'id', 'length'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'ner_tags_str', 'id', 'length'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'ner_tags_str', 'id', 'length'],
        num_rows: 2582
    })
})
{'tokens': List(Value('string')), 'ner_tags': List(Value('int64')), 'ner_tags_str': List(Value('string')), 'id': Value('int64'), 'length': Value('int64')}
Количество строк в train: 7746
dict_keys(['tokens', 'ner_tags', 'ner_tags_str', 'id', 'length'])


Теперь доступ к данным упрощен, цель достигнута. Проверим на какое количество тегов размечен датасет в формате BIO

In [5]:
from tqdm import tqdm

unique_bio = set()

for elem in tqdm(data["train"]['data'][0]):
  for tag in elem['ner_tags_str']:
    unique_bio.add(tag)

unique_bio

100%|██████████| 7746/7746 [00:00<00:00, 207134.71it/s]


{'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'}

Воспользуюсь готовыми методами для нормализации текста, определение граничных индексов каждой отдельной сущности. Также заимствую метод для расчета метрики и классификации ошибок решения NER задачи

In [ ]:
unpacked_dataset['train']['tokens'][2]

['В',
 'Ханты-Мансийском',
 'автономном',
 'округе',
 'с',
 'должности',
 'снят',
 'начальник',
 'УВД',
 'Николай',
 'Гудожников',
 '.']

In [6]:
def normalize_text(s):
    s = s.replace("\u00ad", "") # soft hyphen
    s = s.replace("\u200b", "") # zero-width space
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [7]:
def tokens_to_text_and_offsets(tokens):
    # Собираем текст как " ".join(tokens) и считаем char offsets каждого токена
    parts = []
    offsets = []
    cur = 0
    for i, t in enumerate(tokens):
        if i > 0:
            parts.append(" ")
            cur += 1
        start = cur
        parts.append(t)
        cur += len(t)
        end = cur
        offsets.append((start, end))
    text = "".join(parts)
    text = normalize_text(text)
    return text, offsets

In [8]:
def bio_to_spans(labels):
    spans = []
    start = None
    ent_type = None

    for i, lab in enumerate(labels):
        if lab == "O":
            if ent_type is not None:
                spans.append((start, i, ent_type))
                start, ent_type = None, None
            continue

        prefix, typ = lab.split("-", 1)
        if prefix == "B":
            if ent_type is not None:
                spans.append((start, i, ent_type))
            start, ent_type = i, typ
        elif prefix == "I":
            if ent_type is None:
                # некорректная последовательность: считаем как B
                start, ent_type = i, typ
            elif typ != ent_type:
                spans.append((start, i, ent_type))
                start, ent_type = i, typ

    if ent_type is not None:
        spans.append((start, len(labels), ent_type))
    return spans

In [9]:
def spans_char_from_bio(tokens, offsets, labels):
    # label-спаны в токенах - char-спаны
    spans = bio_to_spans(labels)
    char_spans = []
    for s, e, t in spans:
        cs = offsets[s][0]
        ce = offsets[e-1][1]
        char_spans.append((cs, ce, t))
    return char_spans

In [10]:
def char_spans_to_bio(tokens, offsets, char_spans, label_set=("PER","ORG","LOC")):
    # Строгое отображение: токен должен полностью лежать внутри char-span.
    # Если span обрезает токен - это будет boundary error!
    labels = ["O"] * len(tokens)

    # сортировка по start, длиннее раньше (на случай вложенности)
    char_spans = sorted(char_spans, key=lambda x: (x[0], -(x[1]-x[0])))

    for (cs, ce, typ) in char_spans:
        if typ not in label_set:
            continue
        covered = []
        for i, (ts, te) in enumerate(offsets):
            if ts >= cs and te <= ce:
                covered.append(i)
        if not covered:
            continue
        labels[covered[0]] = "B-" + typ
        for i in covered[1:]:
            labels[i] = "I-" + typ

    return labels

In [11]:
seqeval_metric = evaluate.load("seqeval")

def seqeval_strict_micro_f1(y_true, y_pred):
    # evaluate/seqeval дает entity-level строгое совпадение спана
    # В res есть overall_f1/precision/recall
    res = seqeval_metric.compute(predictions=y_pred, references=y_true, zero_division=0)
    return res

In [12]:
def boundary_error_breakdown(y_true, y_pred):
    # Анализ по спанам: exact, type_confusion, boundary_mismatch, missing, spurious
    stats = {
        "exact": 0,
        "type_confusion": 0,
        "boundary_mismatch": 0,
        "missing": 0,
        "spurious": 0,
        "total_gold": 0,
        "total_pred": 0,
    }

    for gt, pr in zip(y_true, y_pred):
        gsp = bio_to_spans(gt)
        psp = bio_to_spans(pr)

        stats["total_gold"] += len(gsp)
        stats["total_pred"] += len(psp)

        gset = {(s, e, t) for (s, e,  t) in gsp}
        pset = {(s, e, t) for (s, e, t) in psp}

        # Точные совпадения - одинаковые start, end и type
        exact = gset & pset
        stats["exact"] += len(exact)

        # Те же границы, разный тип
        gb = {(s, e): t for (s, e, t) in gsp}
        pb = {(s, e): t for (s, e, t) in psp}
        for b in set(gb.keys()) & set(pb.keys()):
            if gb[b] != pb[b]:
                stats["type_confusion"] += 1

        # boundary mismatch - есть пересечение по токенам, но точного совпадения по границам нет
        def overlaps(a, b):
            (s1, e1, _t1) = a
            (s2, e2, _t2) = b
            return max(s1, s2) < min(e1, e2)

        # Для каждой gold-сущности, которая не попала в exact:
        # если она пересекается хотя бы с одним предсказанным спаном, это boundary_mismatch
        # если не пересекается ни с одним, это missing
        for g in gsp:
            if g in exact:
                continue
            if any(overlaps(g, p) for p in psp):
                stats["boundary_mismatch"] += 1
            else:
                stats["missing"] += 1

        # Для каждой предсказанной сущности, которая не попала в exact:
        # если она не пересекается ни с одной gold-сущностью, это spurious
        # если пересекается, ничего не добавляется, потому что boundary_mismatch
        # уже считают только по gold-стороне, чтобы не удваивать счётчик
        for p in psp:
            if p in exact:
                continue
            if any(overlaps(p, g) for g in gsp):
                pass
            else:
                stats["spurious"] += 1

    return stats

In [13]:
def confusion_matrix_by_type(y_true, y_pred, entity_types=("PER","ORG","LOC")):
    # Матрица только по exact boundary matches + type confusion (по одинаковым границам)
    # Классы: entity_types
    labels = list(entity_types)
    idx = {t:i for i,t in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for gt, pr in zip(y_true, y_pred):
        gsp = bio_to_spans(gt)
        psp = bio_to_spans(pr)
        gb = {(s,e): t for (s,e,t) in gsp}
        pb = {(s,e): t for (s,e,t) in psp}
        for b in set(gb.keys()) & set(pb.keys()):
            g = gb[b]
            p = pb[b]
            if g in idx and p in idx:
                cm[idx[g], idx[p]] += 1

    return cm, labels

def plot_confusion(cm, labels, title):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("pred")
    plt.ylabel("gold")
    plt.show()

Дообучим энкодерную модель для нер задачи, веберу модель от vk RuModernBERT

In [14]:
model_id = "deepvk/RuModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

In [15]:
id2label = {i: l for i, l in enumerate(unique_bio)}
label2id = {l: i for i, l in enumerate(unique_bio)}

Выполним токенезацию с помощью предобученного токенизатора модели, сохраним набор лейблов токенов и флаги расчета атеншена на тот же уровень иерархии датасета, что и остальные данные по сэмплу для удобства

In [16]:
def tokenize_and_align_labels(examples, tokenizer=tokenizer, max_length=256):
    tokenized = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length
    )

    aligned_labels = []
    for i in range(len(examples['tokens'])):
        word_ids = tokenized.word_ids(batch_index=i)
        gold = examples["ner_tags"][i]
        prev = None
        out = []
        for w in word_ids:
            if w is None:
                out.append(-100)
            elif w != prev:
                out.append(gold[w])
            else:
                out.append(-100)
            prev = w
        aligned_labels.append(out)

    tokenized["labels"] = aligned_labels
    return tokenized

In [ ]:
max_length = 192
tokenized = unpacked_dataset.map(lambda x: tokenize_and_align_labels(x, max_length=max_length), batched=True)
tokenized

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_labels = []
        pred_labels = []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_labels.append(id2label[li])
            pred_labels.append(id2label[pi])
        y_true.append(true_labels)
        y_pred.append(pred_labels)

    res = seqeval_metric.compute(predictions=y_pred, references=y_true, zero_division=0)
    return {
        "f1": res["overall_f1"],
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
    }

In [ ]:
data_collator_ner = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="longest")

Сделаем вариант двух этапного дообучения с разморозкой только головы берта. Это должно потенциально повыисить качество модели, так как классификационный слой инициализируется случайно, в отличии от содержащих полезную информацию весов в теле модели, поэтому хаотичные первые обновления весов головы могут заафектить на веса в теле и заставть их обновиться не в оптимальную сторону, что может привести к забыванию моделью части уже выученной информации

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(unique_bio),
    id2label=id2label,
    label2id=label2id
)

# Freeze encoder на быстрый разогрев головы
for p in model.base_model.parameters():
    p.requires_grad = False

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
args = TrainingArguments(
    output_dir="rumodernbert_ner",
    learning_rate=5e-4, # голова обучается быстро
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True, # https://developer.nvidia.com/blog/video-mixed-precision-techniques-tensor-cores-deep-learning/
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer_warmup = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator_ner,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer_warmup.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss,Validation Loss,F1,Precision,Recall
200,0.277613,0.131136,0.813819,0.830351,0.797931
400,0.231340,0.128263,0.816011,0.843005,0.790692
600,0.190110,0.120846,0.840327,0.858105,0.823271
800,0.204389,0.116096,0.832031,0.860597,0.805301
1000,0.179855,0.120867,0.828239,0.868283,0.791726
1200,0.181947,0.109965,0.849397,0.871582,0.828313
1400,0.186029,0.113464,0.840603,0.885866,0.799741
1600,0.167869,0.110964,0.846870,0.878333,0.817582
1800,0.158181,0.109998,0.849147,0.854934,0.843439


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1800, training_loss=0.22182493792639837, metrics={'train_runtime': 359.6915, 'train_samples_per_second': 215.351, 'train_steps_per_second': 6.756, 'total_flos': 3550988888556972.0, 'train_loss': 0.22182493792639837, 'epoch': 7.408247422680413})

Теперь выполним полное дообучение, но уже с пониженным значением LR

In [ ]:
# Unfreeze для полноценного дообучения
for p in model.base_model.parameters():
    p.requires_grad = True

args2 = TrainingArguments(
    output_dir="rumodernbert_ner_ft",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0
)

trainer = Trainer(
    model=model,
    args=args2,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator_ner,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss,F1,Precision,Recall
200,0.085723,0.046312,0.944661,0.949223,0.940142
400,0.042215,0.043707,0.951746,0.972350,0.931997
600,0.012650,0.036063,0.967281,0.963804,0.970782
800,0.006164,0.031077,0.973120,0.972743,0.973497
1000,0.001872,0.035788,0.972374,0.975537,0.969231
1200,0.000699,0.034438,0.973285,0.976324,0.970265
1400,0.001287,0.038976,0.973082,0.976438,0.969748
1600,0.000483,0.037328,0.974751,0.976268,0.973239


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,F1,Precision,Recall
200,0.085723,0.046312,0.944661,0.949223,0.940142
400,0.042215,0.043707,0.951746,0.972350,0.931997
600,0.012650,0.036063,0.967281,0.963804,0.970782
800,0.006164,0.031077,0.973120,0.972743,0.973497
1000,0.001872,0.035788,0.972374,0.975537,0.969231
1200,0.000699,0.034438,0.973285,0.976324,0.970265
1400,0.001287,0.038976,0.973082,0.976438,0.969748
1600,0.000483,0.037328,0.974751,0.976268,0.973239


In [ ]:
preds = trainer.predict(tokenized["test"])
test_metrics = preds.metrics
test_metrics

{'test_loss': 0.030876141041517258,
 'test_f1': 0.9787313899662204,
 'test_precision': 0.9776305923519121,
 'test_recall': 0.9798346693386774,
 'test_runtime': 11.8293,
 'test_samples_per_second': 218.271,
 'test_steps_per_second': 6.847}

In [ ]:
def trainer_predictions_to_seqeval(pred_output):
    logits = pred_output.predictions
    labels = pred_output.label_ids
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_labels = []
        pred_labels = []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_labels.append(id2label[li])
            pred_labels.append(id2label[pi])
        y_true.append(true_labels)
        y_pred.append(pred_labels)
    return y_true, y_pred

y_true_ft, y_pred_ft = trainer_predictions_to_seqeval(preds)

seqeval_strict_micro_f1(y_true_ft, y_pred_ft), boundary_error_breakdown(y_true_ft, y_pred_ft)

({'LOC': {'precision': np.float64(0.9569343065693431),
   'recall': np.float64(0.9493120926864591),
   'f1': np.float64(0.9531079607415487),
   'number': np.int64(1381)},
  'ORG': {'precision': np.float64(0.9855439103722443),
   'recall': np.float64(0.9891186071817193),
   'f1': np.float64(0.9873280231716148),
   'number': np.int64(2757)},
  'PER': {'precision': np.float64(0.9793014230271668),
   'recall': np.float64(0.984139365574623),
   'f1': np.float64(0.9817144339255609),
   'number': np.int64(3846)},
  'overall_precision': np.float64(0.9776305923519121),
  'overall_recall': np.float64(0.9798346693386774),
  'overall_f1': np.float64(0.9787313899662204),
  'overall_accuracy': 0.9947208083407444},
 {'exact': 7823,
  'type_confusion': 52,
  'boundary_mismatch': 66,
  'missing': 95,
  'spurious': 112,
  'total_gold': 7984,
  'total_pred': 8002})

Добавим в тренировочный сет сгенерированные сэмплы данных, собрана они были с помощью модели Deepseek-V3.2

In [ ]:
with open('/content/tokens_add.json', 'r', encoding='utf-8') as f:
    tokens_add = json.load(f)

len(tokens_add)

559

Дообучим модель на MLM задаче, для этого изначельно введем свой PAD токен в deepvk/RuModernBERT-base так как в моделе он отутсвует, а также отсутвует и атрибут eos_token. Попробуем немного расширить данные путем дисциляции 8% процентов датасета из модели Deepseek-V3.2.

In [19]:
tokenizer = AutoTokenizer.from_pretrained("deepvk/RuModernBERT-base")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

print(f"Tokenizer pad_token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")
print(f"Tokenizer vocab size: {len(tokenizer)}")

def preprocess_function(examples):
    texts = [" ".join(tokens) for tokens in examples["tokens"]]
    # texts.extend(tokens_add)
    return tokenizer(texts, truncation=True, padding="max_length", max_length=512)


tokenized_mlm = unpacked_dataset.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=unpacked_dataset["train"].column_names,
)

block_size = 128

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result


lm_dataset = tokenized_mlm.map(group_texts, batched=True, num_proc=4)


model = AutoModelForMaskedLM.from_pretrained("deepvk/RuModernBERT-base")
model.resize_token_embeddings(len(tokenizer))

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir="RuModernBERT_mlm_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
)


print("\nStarting training...")
trainer.train()

Tokenizer pad_token: [PAD] (id: 50283)
Tokenizer vocab size: 50368


Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/7746 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss
1,1.106850,nan
2,1.043468,nan
3,1.010026,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11619, training_loss=1.0827125011207377, metrics={'train_runtime': 4677.986, 'train_samples_per_second': 19.87, 'train_steps_per_second': 2.484, 'total_flos': 7922016659570688.0, 'train_loss': 1.0827125011207377, 'epoch': 3.0})

Сохраним чекпоинт модели на гугл диск для дальнейшего быстрого продолжнения работы

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("checkpoint-11619", 'zip', "/content/RuModernBERT_mlm_model/checkpoint-11619")
files.download("checkpoint-11619.zip")

In [20]:
import gdown
import zipfile
import os

file_id = "1dbK2FWax1rTNjxof1qivFwGaKd1EcTuE"
url = f"https://drive.google.com/uc?id={file_id}"

output = '/content/checkpoint-11619.zip'
gdown.download(url, output, quiet=False)

os.makedirs('/content/checkpoint-11619/', exist_ok=True)

with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall('/content/checkpoint-11619/')

print("Распаковка завершена")

print("\nСодержимое распакованной папки:")
!ls -la /content/checkpoint-11619/

Downloading...
From (original): https://drive.google.com/uc?id=1dbK2FWax1rTNjxof1qivFwGaKd1EcTuE
From (redirected): https://drive.google.com/uc?id=1dbK2FWax1rTNjxof1qivFwGaKd1EcTuE&confirm=t&uuid=c63c7253-4872-47d5-86b3-7ac484a22e5c
To: /content/checkpoint-11619.zip
100%|██████████| 1.66G/1.66G [00:31<00:00, 53.0MB/s]


Распаковка завершена

Содержимое распакованной папки:
total 1758628
drwxr-xr-x 2 root root       4096 Mar 29 11:45 .
drwxr-xr-x 1 root root       4096 Mar 29 11:45 ..
-rw-r--r-- 1 root root       2900 Mar 29 11:45 config.json
-rw-r--r-- 1 root root  598635032 Mar 29 11:45 model.safetensors
-rw-r--r-- 1 root root 1197359627 Mar 29 11:45 optimizer.pt
-rw-r--r-- 1 root root      14645 Mar 29 11:45 rng_state.pth
-rw-r--r-- 1 root root       1465 Mar 29 11:45 scheduler.pt
-rw-r--r-- 1 root root        555 Mar 29 11:45 tokenizer_config.json
-rw-r--r-- 1 root root    4753535 Mar 29 11:45 tokenizer.json
-rw-r--r-- 1 root root      22593 Mar 29 11:45 trainer_state.json
-rw-r--r-- 1 root root       5201 Mar 29 11:45 training_args.bin


Перед финальным дообучением на NER задачу зачистим весь мусор в GPU и удалим старые артифакты модели и тренера. Для честности сравнения оставим такой же парметр max_length

In [21]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

del model
del trainer
del tokenized_mlm

from transformers import AutoModelForMaskedLM, AutoTokenizer

checkpoint_path = "/content/checkpoint-11619"

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint_path,
    num_labels=7,
    ignore_mismatched_sizes=True
)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

max_length = 192
tokenized = unpacked_dataset.map(lambda x: tokenize_and_align_labels(x, tokenizer=tokenizer, max_length=max_length), batched=True)

data_collator_ner = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="longest")


print(f"✅ Model loaded from {checkpoint_path}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: /content/checkpoint-11619
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

✅ Model loaded from /content/checkpoint-11619
GPU memory: 0.02 GB


проверим состояние GPU

In [23]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  17664 KiB |   1783 MiB |  73430 GiB |  73430 GiB |
|       from large pool |  16640 KiB |   1781 MiB |  73291 GiB |  73291 GiB |
|       from small pool |   1024 KiB |      2 MiB |    139 GiB |    139 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  17664 KiB |   1783 MiB |  73430 GiB |  73430 GiB |
|       from large pool |  16640 KiB |   1781 MiB |  73291 GiB |

Теперь обучим дотюненую на MLM задаче модель на NER задачу. Для начала попробую обучить в один этап и с повышенным значением LR для ускорения процесса обучения

In [ ]:
args = TrainingArguments(
    output_dir="rubert_ner",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True, # https://developer.nvidia.com/blog/video-mixed-precision-techniques-tensor-cores-deep-learning/
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer_warmup = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator_ner,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer_warmup.train()

Step,Training Loss,Validation Loss,F1,Precision,Recall
200,0.080293,0.063691,0.983404,0.981976,0.984837
400,0.044907,0.076339,0.982058,0.979173,0.984960
600,0.018332,0.086644,0.984366,0.983265,0.985469
800,0.013719,0.071269,0.985765,0.985023,0.986509
1000,0.007059,0.077997,0.986432,0.986051,0.986814
1200,0.002982,0.083868,0.987442,0.986950,0.987935
1400,0.001802,0.094670,0.987186,0.986824,0.987548
1600,0.000950,0.094589,0.987526,0.986872,0.988180
1800,0.000263,0.099013,0.987576,0.986933,0.988220
2000,0.000180,0.106010,0.987760,0.987077,0.988445


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2430, training_loss=0.02053610627282432, metrics={'train_runtime': 1243.6652, 'train_samples_per_second': 62.284, 'train_steps_per_second': 1.954, 'total_flos': 4794746225603808.0, 'train_loss': 0.02053610627282432, 'epoch': 10.0})

In [ ]:
preds = trainer.predict(tokenized["test"])
test_metrics = preds.metrics
test_metrics

{'test_loss': 0.09275701642036438,
 'test_f1': 0.9893398571442608,
 'test_precision': 0.9891357733639811,
 'test_recall': 0.9895440251572327,
 'test_runtime': 9.1304,
 'test_samples_per_second': 282.791,
 'test_steps_per_second': 8.871}

Хоть и в ходе обучения сформировалась яркая картинка оверфитинга по трейн сет данных. Качество на тесте заметно вырасло по сравнению с моделью до дообучения на MLM. Тестовый f1 вырос на 0,01